## Write a solution to find all dates' id with higher temperatures compared to its previous dates (yesterday).

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Weather table:
+----+------------+-------------+
| id | recordDate | temperature |
+----+------------+-------------+
| 1  | 2015-01-01 | 10          |
| 2  | 2015-01-02 | 25          |
| 3  | 2015-01-03 | 20          |
| 4  | 2015-01-04 | 30          |
+----+------------+-------------+
Output: 
+----+
| id |
+----+
| 2  |
| 4  |
+----+

## taking pandas data from leetcode and converting this into Spark data Frame 


In [57]:
import pandas as pd
data = [[1, '2015-01-01', 10], [2, '2015-01-02', 25], [3, '2015-01-03', 20], [4, '2015-01-04', 30]]
weather = pd.DataFrame(data, columns=['id', 'recordDate', 'temperature']).astype({'id':'Int64', 'recordDate':'datetime64[ns]', 'temperature':'Int64'})

In [58]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("raising temp").getOrCreate()
weather_df=spark.createDataFrame(weather)
weather_df.show()

+---+-------------------+-----------+
| id|         recordDate|temperature|
+---+-------------------+-----------+
|  1|2015-01-01 00:00:00|         10|
|  2|2015-01-02 00:00:00|         25|
|  3|2015-01-03 00:00:00|         20|
|  4|2015-01-04 00:00:00|         30|
+---+-------------------+-----------+



In [66]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag,col,date_diff

window_spec=Window.orderBy(col("recordDate").asc())
df=(weather_df.withColumn("temp_diff",lag("temperature",1)
                         .over(window_spec))
.withColumn("date_diff",lag("recordDate",1).over(window_spec))
)
df1=df.filter((col("temperature")> col("temp_diff")) & (date_diff(col("recordDate"),col("date_diff"))==1)).select("id")
df1.show()


26/06/29 16:45:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/29 16:45:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/29 16:45:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/29 16:45:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/29 16:45:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+
| id|
+---+
|  2|
|  4|
+---+

